# 3. Run a MemAgent on Codex, Claude Code, or OpenHands

This notebook connects the control-plane concepts to production adapters and a real `MemAgent`. It is runnable without credentials: discovery and agent composition are local, while the only vendor execution is protected by an explicit environment flag.

You will learn to:

- probe Codex, Claude Code, OpenHands, and saved-MemAgent capabilities;
- choose between MetaHarness `runtime` and `delegate` composition;
- create a memory-first `MemAgent` whose complete turn runs on a selected harness;
- opt into one safe, read-only vendor call without placing credentials in a notebook; and
- understand how the SDK, CLI, UI, and MCP server expose the same run ledger.

> **Default behavior:** no external agent starts and no model cost is incurred unless you set `MEMORIZZ_RUN_VENDOR_DEMO=1` before launching Jupyter.

## Two valid ways for a MemAgent to use a harness

```mermaid
flowchart TB
    User[User turn] --> Choice{Composition mode}
    Choice -- runtime --> MR[MemoRizz retrieves memory and applies policy]
    MR --> External[External harness owns complete reasoning loop]
    External --> Evidence[Verify, persist, observe, learn]
    Choice -- delegate --> Native[Native MemAgent owns reasoning loop]
    Native --> Tool{Need workspace specialist?}
    Tool -- yes --> Bounded[run_harness_task tool]
    Bounded --> External
    Tool -- no --> NativeResult[Native response]
```

| Mode | Main loop | Model requirement on the `MemAgent` | Best fit |
|---|---|---|---|
| `runtime` | Selected external harness | None; the harness supplies its own model | Coding or terminal task where Codex, Claude Code, or OpenHands should own the full turn |
| `delegate` | Native `MemAgent` | Yes, for native planning and tool selection | General agent that sometimes needs a bounded coding specialist |

In both modes MemoRizz owns scope, retrieved memory, deterministic policy, durable approval checkpoints, normalized events, host verification, and evidence. The recursion guard prevents a MemAgent from routing a turn back into itself.

## 1. Optional vendor prerequisites

Install only the harnesses you operate; MemoRizz does not silently install or vendor their CLIs. Keep authentication in the process environment or each CLI's supported secure configuration—never in notebook cells.

| Harness | Typical command | Authentication / isolation note |
|---|---|---|
| Codex | `codex` | May use its normal `CODEX_HOME` authentication; MemoRizz applies non-interactive sandbox overrides |
| Claude Code | `claude` | MemoRizz uses bare/headless operation; configure `ANTHROPIC_API_KEY` or a supported cloud-provider mode |
| OpenHands | `openhands` | Requires an operator-provided Docker, remote, or sandbox isolation boundary; local headless auto-approval is rejected |

Useful host checks:

```bash
memorizz harness init
memorizz harness doctor
memorizz harness doctor codex
```

To opt into the later execution cell, set values before starting Jupyter:

```bash
export MEMORIZZ_RUN_VENDOR_DEMO=1
export MEMORIZZ_DEMO_HARNESS=codex  # codex | claude-code | openhands
python -m jupyter lab examples/metaharness
```

For OpenHands also configure its isolated wrapper and set `MEMORIZZ_OPENHANDS_EXTERNAL_ISOLATION=true`; choose `MEMORIZZ_DEMO_EXECUTION_BACKEND=docker`, `remote`, or `sandbox`.

In [ ]:
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path
from pprint import pprint

import memorizz
from memorizz import MemAgentBuilder
from memorizz.approval import SQLiteApprovalStore
from memorizz.enums.memory_type import MemoryType
from memorizz.memory_provider.filesystem.provider import FileSystemConfig, FileSystemProvider
from memorizz.metaharness import MetaHarness, SQLiteHarnessRunStore

DEMO_ROOT = Path(tempfile.mkdtemp(prefix="memorizz-metaharness-vendors-"))
WORKSPACE = DEMO_ROOT / "workspace"
WORKSPACE.mkdir()
(WORKSPACE / "hello.py").write_text(
    'def greeting(name: str) -> str:\n'
    '    return f"Hello, {name}!"\n',
    encoding="utf-8",
)
(WORKSPACE / "verify.py").write_text(
    "import ast\nfrom pathlib import Path\nast.parse(Path('hello.py').read_text(encoding='utf-8'))\n",
    encoding="utf-8",
)

provider = FileSystemProvider(
    FileSystemConfig(
        root_path=DEMO_ROOT / "memory",
        embedding_provider=None,
        lazy_vector_indexes=True,
        use_faiss=False,
    )
)

MEMORY_ID = "vendor-harness-tutorial"
USER_ID = "tutorial-user"
THREAD_ID = "hello-review"
REVIEW_QUERY = (
    "Inspect hello.py without modifying any files. Explain the greeting function's "
    "type annotation and exact returned punctuation. Cite any MemoRizz memory "
    "source identifier you rely on."
)
memory_source_id = provider.store(
    {
        "title": "Hello module review rule",
        "content": (
            f"Applicable request: {REVIEW_QUERY}\n"
            "When reviewing hello.py, confirm the function has a string type annotation, "
            "does not mutate the workspace, and describe the exact returned punctuation."
        ),
        "user_id": USER_ID,
        "thread_id": THREAD_ID,
    },
    MemoryType.KNOWLEDGE_BASE,
    memory_id=MEMORY_ID,
)

print({"memorizz_version": memorizz.__version__, "module": memorizz.__file__, "workspace": str(WORKSPACE), "memory_source_id": memory_source_id})

## 2. Build the registry and inspect readiness

`MetaHarness.from_env()` reads secret-free adapter configuration and relevant environment variable names. Probe output is intended for operational decisions and capability reports; it should explain *why* an adapter is not ready rather than failing only after a costly task starts.

```mermaid
flowchart LR
    Requested[Requested harness or auto] --> Probe[Probe every allowlisted adapter]
    Probe --> Available{Executable available?}
    Available -- no --> Reject1[Reject: unavailable]
    Available -- yes --> Policy{Auth, MCP, network, tools, telemetry, isolation fit?}
    Policy -- no --> Reject2[Record rejected reasons]
    Policy -- yes --> Score[Score preference + verified history - cost]
    Score --> Select[Deterministic selected adapter]
```

In [ ]:
service = MetaHarness.from_env(
    memory_provider=provider,
    run_store=SQLiteHarnessRunStore(DEMO_ROOT / "runs.sqlite3"),
    approval_store=SQLiteApprovalStore(DEMO_ROOT / "approvals.sqlite3"),
    allowed_workspace_roots=[str(WORKSPACE)],
)

capabilities = service.list_harnesses()
for item in capabilities:
    print(
        f"{item['name']:<12} ready={str(item['ready']):<5} "
        f"available={str(item['available']):<5} version={item.get('version')}"
    )
    if item.get("error"):
        print("  reason:", item["error"])

assert {"codex", "claude-code", "openhands"}.issubset({item["name"] for item in capabilities})

## 3. Compose a runtime-backed MemAgent

Runtime mode is the literal answer to “can a MemAgent run on a harness?” The `MemAgent` remains the application object and memory identity, but `agent.run()` routes the whole turn into the configured external harness before any native LLM loop begins. Therefore this agent does not need its own `model` for a runtime turn.

The configuration below is deliberately conservative:

- a single temporary workspace root;
- read-only workspace access;
- no network and no MCP tools;
- no inherited secret environment variables; and
- host verification that parses `hello.py` without creating bytecode.

`validate=False` lets the educational notebook compose an agent even when the selected vendor CLI is absent. The opt-in execution cell performs an explicit readiness check before it runs. Production startup should normally validate.

In [ ]:
DEMO_HARNESS = os.getenv("MEMORIZZ_DEMO_HARNESS", "codex").strip().lower().replace("_", "-")
EXECUTION_BACKEND = os.getenv(
    "MEMORIZZ_DEMO_EXECUTION_BACKEND",
    "sandbox" if DEMO_HARNESS == "openhands" else "local",
)
VERIFICATION = f'"{sys.executable}" -B verify.py'

runtime_agent = (
    MemAgentBuilder()
    .with_name("MetaHarness runtime tutorial")
    .with_memory_provider(provider)
    .with_memory_ids(MEMORY_ID)
    .with_execution_harness(
        DEMO_HARNESS,
        meta_harness=service,
        config={
            "workspace": str(WORKSPACE),
            "permissions": {
                "workspace_mode": "read_only",
                "allowed_roots": [str(WORKSPACE)],
                "network": "none",
                "mcp_access": "none",
                "allowed_env": [],
            },
            "budget": {
                "max_wall_time_seconds": 180,
                "max_steps": 12,
                "max_output_tokens": 1_000,
            },
            "verification": {"command": VERIFICATION, "timeout_seconds": 20},
            "metadata": {"execution_backend": EXECUTION_BACKEND},
        },
    )
    .build(validate=False)
)

pprint(
    {
        "agent_id": runtime_agent.agent_id,
        "mode": runtime_agent.meta_harness_mode,
        "default_harness": runtime_agent.default_harness,
        "has_native_llm": runtime_agent.model is not None,
        "memory_ids": runtime_agent.memory_ids,
    }
)
assert runtime_agent.meta_harness_mode == "runtime"
assert runtime_agent.model is None

## 4. Optionally execute one real read-only turn

The following cell is the only one that can invoke an external model. It is gated twice: the environment flag must be true, and the requested adapter's probe must report `ready=True`. No API key or token value is printed.

The task asks the harness to inspect one tiny file and cite MemoRizz memory provenance. After the turn, inspect the durable run rather than trusting only `answer`. Usage and cost precision depend on the selected vendor adapter; missing telemetry is not treated as zero.

In [ ]:
RUN_VENDOR_DEMO = os.getenv("MEMORIZZ_RUN_VENDOR_DEMO", "").strip().lower() in {"1", "true", "yes", "on"}

if not RUN_VENDOR_DEMO:
    print("Safe default: vendor execution skipped. Set MEMORIZZ_RUN_VENDOR_DEMO=1 before starting Jupyter to opt in.")
else:
    capability = service.probe(DEMO_HARNESS)
    if not capability.get("ready"):
        raise RuntimeError(
            f"{DEMO_HARNESS!r} is not ready: {capability.get('error') or 'capability requirements failed'}"
        )

    answer = runtime_agent.run(
        REVIEW_QUERY,
        memory_id=MEMORY_ID,
        user_id=USER_ID,
        thread_id=THREAD_ID,
    )
    latest_run = service.list_runs(limit=1)[0]
    print("Agent answer:", answer)
    pprint(
        {
            "run_id": latest_run["run_id"],
            "status": latest_run["status"],
            "harness": latest_run["harness"],
            "result": latest_run.get("result"),
        }
    )
    assert latest_run["status"] == "succeeded"
    assert latest_run["result"]["verified"] is True
    assert memory_source_id in latest_run["result"]["context_pack"]["source_ids"]

## 5. Compose delegate mode

Delegate mode keeps the native MemAgent in charge and registers three governed tools:

- `list_agent_harnesses` for stable discovery;
- `run_harness_task` for a bounded specialist task; and
- `get_harness_run` for durable evidence lookup.

The write flag and verification command are model-visible task requests, **not approval decisions**. A risky delegated call still pauses in the shared host approval store. In an application, add a configured native LLM with `.with_llm_config(...)`; this notebook omits it so construction remains offline and secret-free.

In [ ]:
delegate_agent = (
    MemAgentBuilder()
    .with_name("MetaHarness delegate tutorial")
    .with_memory_provider(provider)
    .with_memory_ids(MEMORY_ID)
    .with_meta_harness(
        service,
        mode="delegate",
        default_harness="auto",
        config={
            "permissions": {"network": "none", "mcp_access": "none"},
            "budget": {"max_wall_time_seconds": 180, "max_steps": 12},
        },
    )
    .build(validate=False)
)

registered = set(delegate_agent.tool_manager.list_tools())
expected = {"run_harness_task", "get_harness_run", "list_agent_harnesses"}
print("Registered MetaHarness tools:", sorted(registered.intersection(expected)))
assert expected.issubset(registered)
assert delegate_agent.meta_harness_mode == "delegate"

## 6. From execution evidence to continual learning

MetaHarness runs are useful learning evidence because they retain the requested task, selected adapter, context provenance, normalized events, workspace fingerprints, verification, usage, latency, and outcome. They are **not automatically instructions**. MemoRizz's learning control plane should promote only repeated, verified, sufficiently similar trajectories through evidence gates and review.

```mermaid
flowchart LR
    Run[Harness run] --> Ground[Memory source provenance]
    Run --> Trace[Normalized event trace]
    Run --> Verify[Host verification]
    Ground & Trace & Verify --> Evidence[Scoped workflow evidence]
    Evidence --> Canon[Canonicalize repeated paths]
    Canon --> Gates{Quality and diversity gates}
    Gates -- pass --> Shadow[Distill shadow skill]
    Shadow --> Review[Review / evaluate]
    Review -- activate --> Skill[Progressively retrieved skill]
    Review -- reject --> Evidence
```

This matters for instruction hierarchy: an old trajectory is historical evidence, while an activated skill becomes intentional procedural guidance inserted at a controlled role. Separating the two prevents one successful but accidental run from silently rewriting future agent behavior.

## 7. One control plane, four form factors

```mermaid
flowchart TB
    SDK[Python SDK] --> Service[MetaHarness service]
    CLI[MemoRizz CLI] --> Service
    UI[Local UI] --> Service
    MCP[First-party MCP server] --> Service
    Service --> Runs[(Shared run ledger)]
    Service --> Approvals[(Host approval store)]
    Service --> Harnesses[Codex / Claude Code / OpenHands / MemAgent]
```

### CLI

```bash
memorizz harness doctor
memorizz harness run "Inspect hello.py" --workspace "$PWD" --harness codex --mcp-access none --verify "python -m pytest -q"
memorizz harness runs --json
memorizz harness show RUN_ID --events --json
```

### Persist a runtime-backed agent from the CLI

```bash
memorizz agents create --name "Repository maintainer" --no-llm --harness-mode runtime --default-harness codex --harness-workspace "$PWD" --json
```

### UI

Run `memorizz ui`, then open **Agent Harnesses** for capability health, bounded launch, approvals, cancellation, events, diffs, verification, and usage. Configure `MEMORIZZ_UI_AUTH_TOKEN` before any non-local exposure.

### MCP

The first-party server exposes discovery, start, get, list, events, and cancel tools. Remote execution is disabled unless the host explicitly enables it and allowlists workspace roots. Approval/rejection is deliberately absent from model-visible MCP schemas.

```bash
memorizz mcp serve --transport streamable-http --allow-agent-execution --allow-harness-execution --harness-workspace-root /srv/workspaces/project-a
```

## Production checklist

Before moving from this tutorial to a shared environment:

- pin and probe each external CLI version at startup;
- use exact workspace allowlists, clean Git worktrees for direct writes, and disposable workspaces for comparisons;
- keep network at `none` unless the task has a reviewed egress need;
- allowlist environment variable **names**, never persist secret values in tasks or configuration;
- require a real external isolation wrapper for OpenHands;
- configure independent host verification instead of treating the final response as proof;
- authenticate approval and UI surfaces and bind every remote MCP run to its principal;
- monitor missing token/cost telemetry rather than interpreting it as zero;
- retain normalized traces long enough to debug failures, with redaction and tenant scope; and
- admit only verified, business-graded evidence into continual-learning promotion.

You now have the same composition pattern used by the SDK, CLI, local UI, and MemoRizz MCP server—without coupling application code to a single agent vendor.

Continue to notebook 4 to compose Codex and Claude Code as role-specific delegates in one deterministic MemAgent multi-agent workflow.

In [ ]:
delegate_agent.close(close_memory_provider=False)
runtime_agent.close(close_memory_provider=False)
service.close()
provider.close()
shutil.rmtree(DEMO_ROOT, ignore_errors=True)
print("Removed tutorial resources:", DEMO_ROOT)